# Harry potter discovery
#### Progetto per l'insegnamento di Data Analytics dell'anno accademico 2024/2025.
#### Componenti: Alessia Novacco 918850, Nicolò Sansevrino 865889

Il progetto verte su ....

In [172]:
!pip install pandas networkx matplotlib seaborn tensorflow

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


In [173]:
import pandas as pd
import re
import csv
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from transformers import RobertaTokenizer, TFRobertaModel, TFAutoModelForSequenceClassification
import pandas as pd
import numpy as np
import scipy
import re
import csv
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from transformers import RobertaTokenizer, TFRobertaModel, TFAutoModelForSequenceClassification
from tqdm import tqdm


In [174]:
characters = pd.read_csv("Harry_Potter_Movies/Characters.csv", encoding='latin-1')
characters.head()

,Character ID,Character Name,Species,Gender,House,Patronus,Wand (Wood),Wand (Core)
0,1,Harry Potter,Human,Male,Gryffindor,Stag,Holly,Phoenix Feather
1,2,Ron Weasley,Human,Male,Gryffindor,Jack Russell Terrier,NaN,NaN
2,3,Hermione Granger,Human,Female,Gryffindor,Otter,Vine,Dragon Heartstring
3,4,Albus Dumbledore,Human,Male,Gryffindor,Phoenix,Elder,Thestral Tail Hair
4,5,Rubeus Hagrid,Half-Human/Half-Giant,Male,Gryffindor,NaN,Oak,NaN


In [175]:
movies = pd.read_csv("Harry_Potter_Movies/Movies.csv", encoding="utf-8-sig")
movies.head(9)

,Movie ID,Movie Title,Release Year,Runtime,Budget,Box Office
0,1,Harry Potter and the Philosopher's Stone,2001,152,"$125,000,000","$1,002,000,000"
1,2,Harry Potter and the Chamber of Secrets,2002,161,"$100,000,000","$880,300,000"
2,3,Harry Potter and the Prisoner of Azkaban,2004,142,"$130,000,000","$796,700,000"
3,4,Harry Potter and the Goblet of Fire,2005,157,"$150,000,000","$896,400,000"
4,5,Harry Potter and the Order of the Phoenix,2007,138,"$150,000,000","$942,000,000"
5,6,Harry Potter and the Half-Blood Prince,2009,153,"$250,000,000","$943,200,000"
6,7,Harry Potter and the Deathly Hallows Part 1,2010,146,"$200,000,000","$976,900,000"
7,8,Harry Potter and the Deathly Hallows Part 2,2011,130,"$250,000,000","$1,342,000,000"


In [176]:
dialogues = pd.read_csv("Harry_Potter_Movies/Dialogue.csv", encoding='latin-1')
dialogues.head()

,Dialogue ID,Chapter ID,Place ID,Character ID,Dialogue
0,1,1,8,4,I should have known that you would be here...P...
1,2,1,8,7,"Good evening, Professor Dumbledore. Are the ru..."
2,3,1,8,4,"I'm afraid so, Professor. The good, and the bad."
3,4,1,8,7,And the boy?
4,5,1,8,4,Hagrid is bringing him.


In [177]:
chapters = pd.read_csv("Harry_Potter_Movies/Chapters.csv", encoding='latin-1')
chapters.head()

,Chapter ID,Chapter Name,Movie ID,Movie Chapter
0,1,Doorstep Delivery,1,1
1,2,The Vanishing Glass,1,2
2,3,Letters from No One,1,3
3,4,Keeper of the Keys,1,4
4,5,Diagon Alley,1,5


In [178]:
dialogs = dialogues.merge(chapters[["Chapter ID", "Movie ID"]], on="Chapter ID")
dialogs = dialogs.sort_values(by=["Movie ID", "Chapter ID", "Dialogue ID"])
dialogs.head()

,Dialogue ID,Chapter ID,Place ID,Character ID,Dialogue,Movie ID
0,1,1,8,4,I should have known that you would be here...P...,1
1,2,1,8,7,"Good evening, Professor Dumbledore. Are the ru...",1
2,3,1,8,4,"I'm afraid so, Professor. The good, and the bad.",1
3,4,1,8,7,And the boy?,1
4,5,1,8,4,Hagrid is bringing him.,1


In [179]:
# Aggiungi Character Name
dialogs = dialogs.merge(characters[["Character ID", "Character Name"]], on="Character ID")
dialogs["speaker"] = dialogs["Character Name"].str.lower().str.strip()
dialogs.head()

,Dialogue ID,Chapter ID,Place ID,Character ID,Dialogue,Movie ID,Character Name,speaker
0,1,1,8,4,I should have known that you would be here...P...,1,Albus Dumbledore,albus dumbledore
1,2,1,8,7,"Good evening, Professor Dumbledore. Are the ru...",1,Minerva McGonagall,minerva mcgonagall
2,3,1,8,4,"I'm afraid so, Professor. The good, and the bad.",1,Albus Dumbledore,albus dumbledore
3,4,1,8,7,And the boy?,1,Minerva McGonagall,minerva mcgonagall
4,5,1,8,4,Hagrid is bringing him.,1,Albus Dumbledore,albus dumbledore


In [180]:
graphs = {}  # film_id → grafo
centrality_results = {}  # film_id → df con centralità

for mid, g in dialogs.groupby("Movie ID"):

    G = nx.DiGraph()

    # Lista dei turni parlati
    speakers = g["Character Name"].tolist()

    # Costruisci coppie consecutive A→B
    pairs = [(a, b) for a, b in zip(speakers[:-1], speakers[1:]) if a != b]

    # Conta interazioni tra ogni coppia
    interaction_count = defaultdict(int)
    interlocutors = defaultdict(set)

    for a, b in pairs:
        interaction_count[(a, b)] += 1
        interlocutors[a].add(b)

    max_count = max(interaction_count.values())

    # Calcola somma delle interazioni per ogni A (normalizzazione per A)
    for (a, b), count in interaction_count.items():
        norm_weight = count / max_count
        G.add_edge(a, b, weight=norm_weight)


    graphs[mid] = G

    betweenness = nx.betweenness_centrality(G, weight="weight")

    #  Calcolo degree centrality su grafo non orientato temporaneo
    G_undirected = G.to_undirected()
    degree = nx.degree_centrality(G_undirected)

    # Crea DataFrame centralità
    df_centrality = pd.DataFrame({
        'Character Name': list(betweenness.keys()),
        'Betweenness': [betweenness[k] for k in betweenness.keys()],
        'Centrality': [degree[k] for k in betweenness.keys()],
        'Movie ID': mid
    })

    centrality_results[mid] = df_centrality

    # OUTPUT DI CONTROLLO
    print(f"Movie ID {mid} - Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}")
    display(df_centrality.sort_values("Betweenness", ascending=False).head(10))

    centrality_all = pd.concat(centrality_results, ignore_index=True)

    # Elimina le colonne che causano conflitto PRIMA del merge
    dialogs = dialogs.drop(columns=['Centrality', 'Betweenness'], errors='ignore')

    # Ora puoi eseguire il merge tranquillamente
    dialogs = dialogs.merge(centrality_all, on=['Movie ID', 'Character Name'], how='left')

Movie ID 1 - Nodi: 50, Archi: 231


,Character Name,Betweenness,Centrality,Movie ID
6,Harry Potter,0.345273,0.734694,1
2,Rubeus Hagrid,0.224328,0.408163,1
21,Ron Weasley,0.168609,0.408163,1
43,Lee Jordan,0.152056,0.163265,1
23,Hermione Granger,0.136764,0.346939,1
31,Girl,0.127722,0.204082,1
12,Boy,0.089449,0.163265,1
1,Minerva McGonagall,0.081923,0.285714,1
39,Argus Filch,0.073772,0.142857,1
0,Albus Dumbledore,0.070910,0.183673,1


Movie ID 2 - Nodi: 51, Archi: 257


,Character Name,Betweenness,Centrality,Movie ID
0,Harry Potter,0.269975,0.66,2
5,Ron Weasley,0.269841,0.56,2
17,Gilderoy Lockhart,0.161372,0.34,2
15,Hermione Granger,0.151411,0.40,2
18,Draco Malfoy,0.123738,0.32,2
12,Other,0.111129,0.18,2
26,Pomona Sprout,0.102159,0.12,2
23,Severus Snape,0.075395,0.18,2
3,Dudley Dursley,0.069154,0.10,2
24,Albus Dumbledore,0.058013,0.28,2


Movie ID 3 - Nodi: 40, Archi: 199


...

,Character Name,Betweenness,Centrality,Movie ID
10,Hermione Granger,0.299548,0.473684,6
4,Harry Potter,0.235784,0.631579,6
7,Ginny Weasley,0.202361,0.394737,6
15,Draco Malfoy,0.172478,0.315789,6
9,Ron Weasley,0.169443,0.447368,6
5,Albus Dumbledore,0.160812,0.315789,6
6,Horace Slughorn,0.144337,0.421053,6
14,Luna Lovegood,0.094123,0.184211,6
20,Minerva McGonagall,0.073735,0.210526,6
28,Neville Longbottom,0.073709,0.131579,6


Movie ID 7 - Nodi: 57, Archi: 233


,Character Name,Betweenness,Centrality,Movie ID
17,Harry Potter,0.343468,0.607143,7
2,Hermione Granger,0.310606,0.410714,7
6,Voldemort,0.248505,0.285714,7
27,Remus Lupin,0.125322,0.250000,7
4,Ron Weasley,0.117326,0.392857,7
13,Bellatrix Lestrange,0.103107,0.196429,7
42,Dobby,0.097935,0.142857,7
12,Pius Thicknesse,0.083927,0.089286,7
21,Alastor Moody,0.077948,0.142857,7
22,Kingsley Shacklebolt,0.075979,0.107143,7


Movie ID 8 - Nodi: 50, Archi: 173


,Character Name,Betweenness,Centrality,Movie ID
2,Harry Potter,0.537508,0.673469,8
4,Ron Weasley,0.229746,0.346939,8
19,Ginny Weasley,0.206786,0.244898,8
11,Voldemort,0.194696,0.285714,8
15,Seamus Finnigan,0.158539,0.102041,8
5,Hermione Granger,0.140211,0.285714,8
8,Bellatrix Lestrange,0.089531,0.183673,8
21,Severus Snape,0.080272,0.163265,8
36,Sybill Trelawney,0.064413,0.061224,8
14,Neville Longbottom,0.055127,0.204082,8


In [181]:
dialogs.head()

,Dialogue ID,Chapter ID,Place ID,Character ID,Dialogue,Movie ID,Character Name,speaker,Betweenness,Centrality
0,1,1,8,4,I should have known that you would be here...P...,1,Albus Dumbledore,albus dumbledore,0.070910,0.183673
1,2,1,8,7,"Good evening, Professor Dumbledore. Are the ru...",1,Minerva McGonagall,minerva mcgonagall,0.081923,0.285714
2,3,1,8,4,"I'm afraid so, Professor. The good, and the bad.",1,Albus Dumbledore,albus dumbledore,0.070910,0.183673
3,4,1,8,7,And the boy?,1,Minerva McGonagall,minerva mcgonagall,0.081923,0.285714
4,5,1,8,4,Hagrid is bringing him.,1,Albus Dumbledore,albus dumbledore,0.070910,0.183673


In [ ]:
# Usa pesi per le centralità
degree_c = nx.degree_centrality(G)  # NON usa pesi (opzionale)
in_deg   = G.in_degree(weight="weight")
out_deg  = G.out_degree(weight="weight")
pagerank = nx.pagerank(G, weight="weight")

df_c = pd.DataFrame({
    "character": list(G.nodes),
    "in_degree": [in_deg[n] for n in G.nodes],
    "out_degree": [out_deg[n] for n in G.nodes],
    "pagerank": [pagerank[n] for n in G.nodes],
    "raw_degree": [degree_c[n] for n in G.nodes],
    "movie_id": mid
})

centrality_results[mid] = df_c

In [ ]:
centrality_df = pd.concat(centrality_results.values(), ignore_index=True)

In [ ]:
# Funzione per disegnare il grafo di un tuo film reale
def draw_film_graph_real(G, cut_threshold, pagerank=None, title="Grafo delle interazioni normalizzate tra personaggi", k=0.6):
    # stampiamo il pesoo degli archi in ordine decrescente
    '''print("Edge weights (sorted):")
    edge_weights = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
    for u, v, d in edge_weights:
        print(f"{u} → {v}: {d['weight']:.2f}")'''

    if pagerank is None:
        pagerank = nx.pagerank(G, weight='weight')

    pos = nx.spring_layout(G, seed=42, k=0.6)
    fig, ax = plt.subplots(figsize=(14, 10))


    node_sizes = [15000 * pagerank[n] for n in G.nodes()]
    edge_sizes = [20 * G[u][v]['weight'] for u, v in G.edges()]
    # impostiamo una edge size massima per evitare che le linee siano troppo spesse
    max_edge_size = 2
    edge_sizes = [min(size, max_edge_size) for size in edge_sizes]
    # Normalizziamo i colori dei nodi in base al pagerank
    node_colors = [pagerank[n] for n in G.nodes()]
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)} # pesi delle interazioni, per ora le conserviamo e basta
    # creiamo degli alfa per gli archi variabili in base al peso dell'arco in modo che siano =0.2 se il loro peso è sotto il threshold, altrimenti =1

    edge_alphas = [0.2 if G[u][v]['weight'] < cut_threshold else 1 for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, cmap=plt.cm.plasma, alpha=0.9, edgecolors='black', ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_sizes, alpha=edge_alphas, arrows=True, edge_color='#012201', ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_color='black', ax=ax)
    #nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='black', font_size=8, ax=ax)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.plasma, norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors)))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Pagerank")

    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # mostriamo edge_labels_df in ordine decrescente per peso
    edge_labels_df = pd.DataFrame(edge_labels.items(), columns=['Edge', 'Weight'])
    edge_labels_df = edge_labels_df.sort_values(by='Weight', ascending=False)
    #print("Edge labels (sorted by weight):")
    #print(edge_labels_df.head(10))


In [ ]:

def draw_heatmap(adj_matrix, cut_threshold):
  plt.figure(figsize=(12, 10))
  mask = adj_matrix <= cut_threshold
  ax = sns.heatmap(adj_matrix, cmap="YlOrRd", linewidths=0.5, linecolor='gray', mask=mask, cbar=True)

  plt.title("Heatmap delle interazioni tra personaggi")
  plt.xlabel("Personaggio")
  plt.ylabel("Personaggio")

  # grassetto per nomi asse X
  for i, label in enumerate(ax.get_xticklabels()):
      name = adj_matrix.columns[i]
      if (adj_matrix.iloc[:, i] > cut_threshold).any():
          label.set_fontweight('bold')
      label.set_text(name)
      label.set_rotation(90)

  # grassetto per nomi asse Y
  for i, label in enumerate(ax.get_yticklabels()):
      name = adj_matrix.index[i]
      if (adj_matrix.iloc[i, :] > cut_threshold).any():
          label.set_fontweight('bold')
      label.set_text(name)
      label.set_rotation(0)

  # Aggiunta bordi spessi per celle con valore > cut_threshold
  for i in range(adj_matrix.shape[0]):
      for j in range(adj_matrix.shape[1]):
          value = adj_matrix.iloc[i, j]
          if value > cut_threshold:
              # disegna un rettangolo con bordo spesso intorno alla cella
              ax.add_patch(plt.Rectangle(
                  (j, i), 1, 1, fill=False, edgecolor='black', lw=1))

  ax.set_xticklabels(ax.get_xticklabels())
  ax.set_yticklabels(ax.get_yticklabels())

  plt.tight_layout()
  plt.show()


In [ ]:
def study_film(film_id, threshold):
  G = graphs[film_id]
  pagerank = nx.pagerank(G, weight="weight")  

  draw_film_graph_real(G, threshold, pagerank, title=f"Grafo delle interazioni normalizzate - Film {film_id}")

  betweenness = nx.betweenness_centrality(G, weight='weight')
  degree = nx.degree_centrality(G)

  adj_matrix = nx.to_pandas_adjacency(G, weight='weight')

  centrality_df = pd.DataFrame({
      "Character Name": list(G.nodes()),
      "Betweenness": [betweenness[n] for n in G.nodes()],
      "Degree": [degree[n] for n in G.nodes()]
  })

  draw_heatmap(adj_matrix, threshold)
  #print(centrality_df)

In [ ]:
for i in range(1, 9):
  study_film(i, threshold=0.15) # giocare con la soglia di taglio

### PARTE 1: 
#### Identificare quali personaggi hanno maggiore intensità emotiva senza essere centrali per possibili prequel o spin off

In [ ]:
!pip install vaderSentiment transformers torch networkx scipy --quiet
!pip install -q networkx pandas matplotlib

In [ ]:
# Merge dialoghi con info capitolo e film
dialogs = dialogues.merge(chapters[["Chapter ID", "Movie ID"]], on="Chapter ID")
dialogs = dialogs.merge(characters[["Character ID", "Character Name"]], on="Character ID")
dialogs = dialogs.sort_values(by=["Movie ID", "Chapter ID", "Dialogue ID"])
dialogs["speaker"] = dialogs["Character Name"].str.lower().str.strip()
dialogs["dialogue"] = dialogs["Dialogue"].astype(str)

In [ ]:
dialogs.head()

In [ ]:
import pandas as pd
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import entropy
import networkx as nx

# --- VADER SENTIMENT
analyzer = SentimentIntensityAnalyzer()

def vader_score(text):
    try:
        s = analyzer.polarity_scores(str(text))
        return pd.Series([s['compound'], s['pos'], s['neu'], s['neg']])
    except:
        return pd.Series([0, 0, 1, 0])

dialogs[['vader_compound', 'vader_pos', 'vader_neu', 'vader_neg']] = dialogs['dialogue'].apply(vader_score)

# --- BERT SENTIMENT (NLPTown BERT: label 1-5)
bert_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=-1)
dialogs['bert_sentiment'] = dialogs['dialogue'].apply(lambda x: int(bert_sentiment(x)[0]['label'][0]))

# --- Entropia emozionale
def calc_vader_entropy(series):
    binned = pd.cut(series, bins=[-1.01, -0.2, 0.2, 1.01], labels=['neg', 'neu', 'pos'])
    probs = binned.value_counts(normalize=True)
    return entropy(probs)

def calc_bert_entropy(series):
    # 1-2 = neg, 3 = neu, 4-5 = pos
    binned = pd.cut(series, bins=[0.5, 2.5, 3.5, 5.5], labels=['neg', 'neu', 'pos'])
    probs = binned.value_counts(normalize=True)
    return entropy(probs)

vader_entropies = dialogs.groupby(['Movie ID', 'speaker']).apply(
    lambda x: calc_vader_entropy(x['vader_compound'])
).reset_index().rename(columns={0: 'vader_entropy'})

bert_entropies = dialogs.groupby(['Movie ID', 'speaker']).apply(
    lambda x: calc_bert_entropy(x['bert_sentiment'])
).reset_index().rename(columns={0: 'bert_entropy'})

entropy_df = vader_entropies.merge(bert_entropies, on=['Movie ID', 'speaker'])

# --- Analisi centralità sui grafi delle interazioni
centrality_list = []
for movie_id, group in dialogs.groupby('Movie ID'):
    G = nx.DiGraph()
    speakers = group['speaker'].tolist()
    edges = [(a, b) for a, b in zip(speakers[:-1], speakers[1:]) if a != b]
    G.add_edges_from(edges)
    out_deg = G.out_degree()
    in_deg = G.in_degree()
    pagerank = nx.pagerank(G, weight=None)
    temp = pd.DataFrame({
        'speaker': list(G.nodes()),
        'out_degree': [out_deg[n] for n in G.nodes()],
        'in_degree': [in_deg[n] for n in G.nodes()],
        'pagerank': [pagerank[n] for n in G.nodes()],
        'Movie ID': movie_id
    })
    centrality_list.append(temp)
centrality_df = pd.concat(centrality_list, ignore_index=True)

# --- Merge delle feature
features = entropy_df.merge(centrality_df, on=['Movie ID', 'speaker'])

# --- Filtra i personaggi "secondari ma vivi"
pg_mean = features['pagerank'].mean()
deg_mean = features['out_degree'].mean()
entr_mean = features['bert_entropy'].mean()

targets = (
    features[
        (features['pagerank'] <= pg_mean) &
        (features['out_degree'] >= deg_mean) &
        (features['bert_entropy'] >= entr_mean)
    ]
    .sort_values(['Movie ID', 'bert_entropy'], ascending=[True, False])
)

print("Totale features:", len(features))
print("Dopo filtro pagerank:", len(features[features['pagerank'] <= pg_mean]))
print("Dopo filtro out_degree:", len(features[features['out_degree'] >= deg_mean]))
print("Dopo filtro entropy:", len(features[features['bert_entropy'] >= entr_mean]))

print(dialogs['bert_sentiment'].value_counts())
print(dialogs[['speaker', 'bert_sentiment']].groupby('speaker').nunique().sort_values('bert_sentiment'))

# --- STEP 5: NORMALIZZA E MERGE PER INFORMAZIONI ---
characters['speaker_lower'] = characters['Character Name'].str.lower().str.strip()
targets = targets.copy()
targets['speaker_lower'] = targets['speaker'].str.lower().str.strip()
tabella = targets.merge(characters, on='speaker_lower', how='left')

# --- STEP 6: AGGREGA SU TUTTA LA SAGA ---
agg_tabella = (
    tabella
    .groupby('speaker_lower')
    .agg(
        vader_entropy_mean=('vader_entropy', 'mean'),
        bert_entropy_mean=('bert_entropy', 'mean'),
        out_degree_mean=('out_degree', 'mean'),
        in_degree_mean=('in_degree', 'mean'),
        pagerank_mean=('pagerank', 'mean'),
        Character_Name=('Character Name', 'first'),
        num_films=('Movie ID', 'nunique')
    )
    .reset_index()
)

# Normalizza le colonne di interesse tra 0 e 1
for col in ['bert_entropy_mean', 'out_degree_mean', 'pagerank_mean']:
    min_val, max_val = agg_tabella[col].min(), agg_tabella[col].max()
    agg_tabella[f'norm_{col}'] = (agg_tabella[col] - min_val) / (max_val - min_val)

# Calcola ranking combinato: puoi cambiare i pesi!
agg_tabella['score'] = (
    agg_tabella['norm_bert_entropy_mean'] * 0.5 +
    agg_tabella['norm_out_degree_mean'] * 0.5 -
    agg_tabella['norm_pagerank_mean'] * 0.8
)

# --- STEP 8: ORDINA PER SCORE ---
agg_tabella = agg_tabella.sort_values(by='score', ascending=False)

# --- STEP 9: VISUALIZZA I MIGLIORI CANDIDATI "SECONDARI ATTIVI" ---
display(agg_tabella.head(15))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Colonne da visualizzare
plot_cols = ['Character_Name', 'bert_entropy_mean', 'out_degree_mean', 'in_degree_mean', 'pagerank_mean']
plot_df = agg_tabella.head(15)[plot_cols].set_index('Character_Name')

plt.figure(figsize=(10, 7))
sns.heatmap(plot_df, annot=True, fmt=".2f", cmap='YlGnBu', cbar_kws={'label': 'Valore'})
plt.title("Heatmap delle metriche dei migliori candidati 'secondari attivi'")
plt.ylabel("Personaggio")
plt.xlabel("Metrica")
plt.tight_layout()
plt.show()

In [ ]:
# Lista dei top N personaggi (già calcolata sopra)
top_n = 10
best_characters = agg_tabella.head(top_n)['speaker_lower'].tolist()

from transformers import pipeline

bert_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=-1)
dialogs['bert_sentiment'] = dialogs['dialogue'].apply(lambda x: int(bert_sentiment(x)[0]['label'][0]))

# Ricalcola entropia BERT per ogni personaggio e film
def bert_entropy_for_group(df):
    # 1-2 = neg, 3 = neu, 4-5 = pos
    binned = pd.cut(df['bert_sentiment'], bins=[0.5, 2.5, 3.5, 5.5], labels=['neg', 'neu', 'pos'])
    probs = binned.value_counts(normalize=True)
    return entropy(probs)

entropy_line = (
    dialogs
    .copy()
    .assign(speaker_lower=lambda d: d['speaker'].str.lower().str.strip())
    .query('speaker_lower in @best_characters')
    .groupby(['Movie ID', 'speaker_lower'])
    .apply(bert_entropy_for_group)
    .reset_index(name='bert_entropy')
)

# Unisci i nomi "umani"
entropy_line = entropy_line.merge(
    agg_tabella[['speaker_lower', 'Character_Name']], on='speaker_lower', how='left'
)

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

for name, group in entropy_line.groupby('Character_Name'):
    plt.plot(group['Movie ID'], group['bert_entropy'], marker='o', label=name)

plt.xlabel('Movie ID')
plt.ylabel('Entropia BERT')
plt.title('Andamento Entropia (BERT) nei migliori personaggi secondari attivi')
plt.legend(title='Personaggio', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

for char in entropy_line['Character_Name'].unique():
    char_data = entropy_line[entropy_line['Character_Name'] == char].sort_values('Movie ID')

    plt.figure(figsize=(7,4))
    plt.plot(char_data['Movie ID'], char_data['bert_entropy'], marker='o')
    plt.title(f"Andamento entropia BERT - {char}")
    plt.xlabel('Movie ID')
    plt.ylabel('Entropia BERT')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Pivot: righe=Personaggi, colonne=Film, valori=entropia BERT
heatmap_df = entropy_line.pivot(index='Character_Name', columns='Movie ID', values='bert_entropy')

import seaborn as sns

plt.figure(figsize=(10, 7))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap='YlGnBu', cbar_kws={'label': 'Entropia BERT'})
plt.title("Heatmap entropia BERT (personaggi vs film)")
plt.ylabel("Personaggio")
plt.xlabel("Movie ID")
plt.tight_layout()
plt.show()


### PARTE 2: 
#### Studiare i pattern emotivi dei personaggi centrali e utilizzare i loro studi per crearne pattern per altri franchise

In [ ]:
# 1. Merge TUTTI I DATI
dialogs = dialogues.merge(chapters[["Chapter ID", "Movie ID"]], on="Chapter ID")
dialogs = dialogs.merge(characters[["Character ID", "Character Name"]], on="Character ID")
dialogs = dialogs.sort_values(by=["Movie ID", "Chapter ID", "Dialogue ID"])
dialogs["speaker"] = dialogs["Character Name"].str.lower().str.strip()
dialogs["dialogue"] = dialogs["Dialogue"].astype(str)

# 2. Estrazione emozione con BERT (batch!)
from transformers import pipeline
from tqdm.auto import tqdm

emotion_model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)
batch_size = 32
texts = dialogs['dialogue'].tolist()
emotions = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    preds = emotion_model(batch)
    for pred in preds:
        label = pred[0]['label'].lower()
        emotions.append(label)

label_map = {
    'joy': 'joy',
    'anger': 'anger',
    'fear': 'fear',
    'sadness': 'sadness',
    'love': 'love',
    'surprise': 'surprise',
}
dialogs['Emotion'] = pd.Series(emotions).map(label_map)
dialogs = dialogs[dialogs['Emotion'].isin(label_map.values())]
dialogs['Emotion'] = pd.Categorical(dialogs['Emotion'], categories=list(label_map.values()))

# 3. Il DataFrame df_emotions ora è pronto
df_emotions = dialogs.copy()


In [ ]:
df_emotions.head()

In [ ]:
from IPython.display import display, Markdown
# --- Uniforma i nomi
centrality_df['speaker'] = centrality_df['speaker'].str.lower().str.strip()
df_emotions['Character Name'] = df_emotions['Character Name'].str.lower().str.strip()

# --- Conta presenza in quanti film
film_count = centrality_df.groupby('speaker')['Movie ID'].nunique()
min_films = 2
relevant_speakers = film_count[film_count >= min_films].index

# --- Pagerank medio su personaggi rilevanti
pagerank_by_char = (
    centrality_df[centrality_df['speaker'].isin(relevant_speakers)]
    .groupby('speaker')['pagerank']
    .mean()
    .sort_values(ascending=False)
)
top_n = 10
central_characters = pagerank_by_char.head(top_n).index.tolist()

# --- Tabella elegante per notebook
centrality_table = pagerank_by_char.head(top_n).reset_index()
centrality_table.columns = ['Personaggio', 'Pagerank medio']
centrality_table['Num Film'] = centrality_table['Personaggio'].map(film_count)
display(Markdown("**Top personaggi centrali (per Pagerank, almeno 2 film):**"))
display(centrality_table.style.background_gradient(cmap="Blues")
        .format({'Pagerank medio': '{:.4f}'})
        .bar('Num Film', color='#ffd700')
       )



In [ ]:
for character in central_characters:
    char_data = df_emotions[df_emotions['Character Name'] == character]
    plt.figure(figsize=(10, 5))
    sns.countplot(data=char_data, x='Movie ID', hue='Emotion', order=sorted(char_data['Movie ID'].unique()))
    plt.title(f"Andamento delle emozioni per {character} nei vari film")
    plt.xlabel('Film')
    plt.ylabel('Conteggio emozioni')
    plt.legend(title='Emozione', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()


In [ ]:
for character in central_characters:
    char_data = df_emotions[df_emotions['Character Name'] == character]
    for movie in sorted(char_data['Movie ID'].unique()):
        movie_data = char_data[char_data['Movie ID'] == movie]
        plt.figure(figsize=(14, 5))
        sns.countplot(data=movie_data, x='Chapter ID', hue='Emotion', order=sorted(movie_data['Chapter ID'].unique()))
        plt.title(f"{character} - Emozioni nei capitoli di '{movie}'")
        plt.xlabel('Capitolo')
        plt.ylabel('Conteggio emozioni')
        plt.legend(title='Emozione', bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.show()


In [ ]:
# Pivot per heatmap: (Character, Movie, Chapter) → Emotion count
pivot = (
    df_emotions[df_emotions['Character Name'].isin(central_characters)]
    .groupby(['Character Name', 'Movie ID', 'Chapter ID', 'Emotion'])
    .size()
    .reset_index(name='Count')
)

for movie in sorted(df_emotions['Movie ID'].unique()):
    movie_pivot = pivot[pivot['Movie ID'] == movie]
    # Crea matrice personaggio x emozione (sommata su tutti i capitoli del film)
    emotion_matrix = movie_pivot.pivot_table(index='Character Name', columns='Emotion', values='Count', aggfunc='sum', fill_value=0)
    plt.figure(figsize=(8, 6))
    sns.heatmap(emotion_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Heatmap delle emozioni nei personaggi centrali in '{movie}'")
    plt.xlabel('Emozione')
    plt.ylabel('Personaggio')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

top_n = 5
central_characters = df_emotions['Character Name'].value_counts().head(top_n).index.tolist()
emotion_order = ["sadness", "love", "joy", "anger", "fear", "surprise"]

# Palette: sadness=azzurro, love=rosa, joy=giallo, anger=rosso, fear=arancione, surprise=viola
custom_palette = [
    "#63b3ed",  # sadness: azzurro
    "#e687b5",  # love: rosa
    "#ffe066",  # joy: giallo
    "#e74c3c",  # anger: rosso
    "#ff9900",  # fear: arancione
    "#b084cc"   # surprise: viola/lilla
]

for movie in sorted(df_emotions['Movie ID'].unique()):
    plt.figure(figsize=(len(central_characters)*3, 7))
    heat_data = []
    chapters = sorted(df_emotions[df_emotions['Movie ID'] == movie]['Chapter ID'].unique())
    for character in central_characters:
        row = []
        for chapter in chapters:
            subset = df_emotions[
                (df_emotions['Movie ID'] == movie) &
                (df_emotions['Chapter ID'] == chapter) &
                (df_emotions['Character Name'] == character)
            ]
            if len(subset) == 0:
                row.append(np.nan)
            else:
                dominant_emotion = subset['Emotion'].value_counts().idxmax()
                row.append(emotion_order.index(dominant_emotion))
        heat_data.append(row)
    ax = sns.heatmap(
        np.array(heat_data), 
        cmap=custom_palette, 
        cbar=True, 
        vmin=0, vmax=len(emotion_order)-1,
        linewidths=0.5, 
        annot=False,
        yticklabels=central_characters, 
        xticklabels=chapters
    )
    cbar = ax.collections[0].colorbar
    cbar.set_ticks(np.arange(len(emotion_order)) + 0.5)
    cbar.set_ticklabels(emotion_order)
    plt.title(f"Emozione predominante per film - {movie}")
    plt.xlabel("Capitolo")
    plt.ylabel("Personaggio centrale")
    plt.show()


In [ ]:
for movie in sorted(df_emotions['Movie ID'].unique()):
    chapters = sorted(df_emotions[df_emotions['Movie ID'] == movie]['Chapter ID'].unique())
    for character in central_characters:
        subset = df_emotions[
            (df_emotions['Movie ID'] == movie) &
            (df_emotions['Character Name'] == character)
        ]
        if subset.empty:
            continue
        # Conta emozioni per capitolo
        pivot = subset.groupby(['Chapter ID', 'Emotion']).size().unstack(fill_value=0)[emotion_order]
        pivot = pivot.reindex(chapters, fill_value=0)
        plt.figure(figsize=(15, 6))
        for emotion in emotion_order:
            plt.plot(pivot.index, pivot[emotion], label=emotion)
        plt.title(f"Andamento delle emozioni per {character} in '{movie}'")
        plt.xlabel("Capitolo")
        plt.ylabel("Conteggio")
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
def plot_emotions_all_chapters(df_emotions, character, emotion_order=None):
    df = df_emotions[df_emotions['Character Name'].str.lower() == character.lower()]
    df = df.sort_values(['Movie ID', 'Chapter ID'])
    df['Film-Cap'] = df['Movie ID'].astype(str) + '-' + df['Chapter ID'].astype(str)
    chapters = df['Film-Cap'].unique()
    if emotion_order is None:
        emotion_order = df['Emotion'].unique()
    pivot = df.groupby(['Film-Cap', 'Emotion']).size().unstack(fill_value=0)[emotion_order]
    pivot = pivot.reindex(chapters, fill_value=0)
    plt.figure(figsize=(max(15, int(len(chapters)/2)), 6))
    for emotion in emotion_order:
        plt.plot(
            pivot.index, pivot[emotion], label=emotion, linewidth=3
        )
    plt.title(f"Andamento delle emozioni per {character.title()} (tutti i film, capitolo per capitolo)")
    plt.xlabel("Film-Capitolo")
    plt.ylabel("Conteggio emozioni")
    plt.xticks(rotation=90, fontsize=8)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_emotions_per_film(df_emotions, character, emotion_order=None):
    df = df_emotions[df_emotions['Character Name'].str.lower() == character.lower()]
    df = df.sort_values(['Movie ID'])
    if emotion_order is None:
        emotion_order = df['Emotion'].unique()
    pivot = df.groupby(['Movie ID', 'Emotion']).size().unstack(fill_value=0)[emotion_order]
    pivot = pivot.reindex(sorted(df['Movie ID'].unique()), fill_value=0)
    plt.figure(figsize=(10, 5))
    for emotion in emotion_order:
        plt.plot(
            pivot.index, pivot[emotion], label=emotion, linewidth=3
        )
    plt.title(f"Andamento delle emozioni per {character.title()} (per film)")
    plt.xlabel("Film")
    plt.ylabel("Conteggio emozioni")
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
for character in central_characters:
    plot_emotions_all_chapters(df_emotions, character, emotion_order=['joy','anger','fear','sadness','love','surprise'])
    plot_emotions_per_film(df_emotions, character, emotion_order=['joy','anger','fear','sadness','love','surprise'])


# Parte 3

In questa parte intendiamo fare un'analisi sullo stile emotivo dei personaggi usando la casata di appartenenza come criterio di suddivisione. Perciò isoliamo i personaggi, rappresentiamoli nuovamente su grafi opportunamente divisi e usiamo nuovamente transformers e lexicon e cerchiamo se esistono dei pattern.

In [ ]:
dialogues

In [ ]:
characters

In [ ]:
# mergiamo dialogues con characters 
dialogues = dialogues.merge(characters, on="Character ID")

# eliminiamo le colonne che non ci servono
full_saga_df = dialogues.drop(columns=['Dialogue ID', 'Place ID', 'Patronus', 'Wand (Wood)', 'Wand (Core)'], errors='ignore')

full_saga_df

In [ ]:
# ora creiamo un grafo come in precedenza però sul dataset full_saga_df in modo da fare valutazioni "globali"
G = nx.DiGraph()
# Lista dei turni parlati
speakers = full_saga_df["Character Name"].tolist()
# Costruisci coppie consecutive A→B
pairs = [(a, b) for a, b in zip(speakers[:-1], speakers[1:]) if a != b]
# Conta interazioni tra ogni coppia
interaction_count = defaultdict(int)
interlocutors = defaultdict(set)
for a, b in pairs:
    interaction_count[(a, b)] += 1
    interlocutors[a].add(b)
max_count = max(interaction_count.values())
# Calcola somma delle interazioni per ogni A (normalizzazione per A)
for (a, b), count in interaction_count.items():
    norm_weight = count / max_count
    G.add_edge(a, b, weight=norm_weight)

# aggiungiamo l'attributo House ai nodi
for node in G.nodes():
    # Trova la casa del personaggio
    house = full_saga_df[full_saga_df['Character Name'] == node]['House'].values
    if house.size > 0:
        G.nodes[node]['House'] = house[0]
    else:
        G.nodes[node]['House'] = 'Not a Hogwart Student'

# Calcola centralità
betweenness = nx.betweenness_centrality(G, weight="weight")
# Calcolo degree centrality su grafo non orientato temporaneo
G_undirected = G.to_undirected()
degree = nx.degree_centrality(G_undirected)
# Crea DataFrame centralità
centrality_df = pd.DataFrame({
    'Character Name': list(betweenness.keys()),
    'Betweenness': [betweenness[k] for k in betweenness.keys()],
    'Centrality': [degree[k] for k in betweenness.keys()]
})
# Merge con resized_dialogs
resized_dialogs = full_saga_df.merge(centrality_df, on='Character Name', how='left')

degree_c = nx.degree_centrality(G)  # NON usa pesi (opzionale)
in_deg   = G.in_degree(weight="weight")
out_deg  = G.out_degree(weight="weight")
pagerank = nx.pagerank(G, weight="weight")

df_c = pd.DataFrame({
    "character": list(G.nodes),
    "in_degree": [in_deg[n] for n in G.nodes],
    "out_degree": [out_deg[n] for n in G.nodes],
    "pagerank": [pagerank[n] for n in G.nodes],
    "raw_degree": [degree_c[n] for n in G.nodes],
    "movie_id": mid
})

centrality_results[mid] = df_c

centrality_df = pd.concat(centrality_results.values(), ignore_index=True)

In [ ]:
def draw_saga_graph(G, cut_threshold, pagerank=None, title="Grafo delle interazioni normalizzate tra personaggi", k=0.6):
    if pagerank is None:
        pagerank = nx.pagerank(G, weight='weight')

    fig, ax = plt.subplots(figsize=(14, 10))

    # ----------- LAYOUT RAGGRUPPATO PER CASA -----------
    house_positions = {
        'Gryffindor': np.array([0.5, 0.5]),      
        'Slytherin':  np.array([-0.2, 1.2]),      
        'Ravenclaw':  np.array([1.2, 1.2]),      
        'Hufflepuff': np.array([-0.2, -0.2]),      
        'Unknown':    np.array([1.2, -0.2])
    }

    pos = {}
    rng = np.random.default_rng(seed=42)
    for node in G.nodes():
        house = G.nodes[node].get('House', 'Unknown')
        base_pos = house_positions.get(house, house_positions['Unknown'])
        jitter = rng.normal(loc=0, scale=0.25, size=2)  
        pos[node] = base_pos + jitter

    # ----------------------------------------------------

    node_sizes = [5000 * np.sqrt(pagerank[n]) for n in G.nodes()]
    edge_sizes = [min(20 * G[u][v]['weight'], 2) for u, v in G.edges()]
    
    node_colors = []
    for n in G.nodes():
        house = G.nodes[n].get('House')
        if house == 'Gryffindor':
            node_colors.append('#a6332e')
        elif house == 'Hufflepuff':
            node_colors.append('#efbc2f')
        elif house == 'Ravenclaw':
            node_colors.append('#3c4e91')
        elif house == 'Slytherin':
            node_colors.append('#366447')
        else:
            node_colors.append('#FFFFFF')

    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
    edge_alphas = [0.2 if G[u][v]['weight'] < cut_threshold else 1 for u, v in G.edges()]

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9, edgecolors='black', ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_sizes, alpha=edge_alphas, arrows=True, edge_color='#012201', ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_color='black', ax=ax)

    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Mostra top archi
    edge_labels_df = pd.DataFrame(edge_labels.items(), columns=['Edge', 'Weight'])
    edge_labels_df = edge_labels_df.sort_values(by='Weight', ascending=False)


In [ ]:
def study_saga(threshold):
  pagerank = nx.pagerank(G, weight="weight")  
  draw_saga_graph(G, threshold, pagerank=pagerank, title="Grafo delle interazioni dei personaggi divisi per casa", k=0.2)
  betweenness = nx.betweenness_centrality(G, weight='weight')
  degree = nx.degree_centrality(G)

  adj_matrix = nx.to_pandas_adjacency(G, weight='weight')

  centrality_df = pd.DataFrame({
      "Character Name": list(G.nodes()),
      "Betweenness": [betweenness[n] for n in G.nodes()],
      "Degree": [degree[n] for n in G.nodes()]
  })


In [ ]:
study_saga(threshold=0.05)  # giocare con la soglia di taglio

In [ ]:
gryffindor = full_saga_df[full_saga_df['House'] == 'Gryffindor']
hufflepuff = full_saga_df[full_saga_df['House'] == 'Hufflepuff']
ravenclaw = full_saga_df[full_saga_df['House'] == 'Ravenclaw']      
slytherin = full_saga_df[full_saga_df['House'] == 'Slytherin']
others = full_saga_df[~full_saga_df['House'].isin(['Gryffindor', 'Hufflepuff', 'Ravenclaw', 'Slytherin'])]

gryffindor

In [ ]:
# estriamo i personaggi per casa
houses = {
    'Gryffindor': gryffindor,
    'Hufflepuff': hufflepuff,
    'Ravenclaw': ravenclaw,
    'Slytherin': slytherin
}
gryffindor_characters = gryffindor['Character Name'].unique().tolist()
hufflepuff_characters = hufflepuff['Character Name'].unique().tolist()
ravenclaw_characters = ravenclaw['Character Name'].unique().tolist()
slytherin_characters = slytherin['Character Name'].unique().tolist()
others_characters = others['Character Name'].unique().tolist()

gryffindor_characters

### 3.1 Transformers: sentiment analysis sulle casate con Roberta

Usiamo bhadresh-savani/roberta-base-emotion.
Gestisce bene le parole "inventate" ed è un ottimo modello per il task di sentiment analysis. 

Step 1: verifico la lunghezza dei dialoghi

In [ ]:
# controlliamo quanto sono lunghi tutti i dialoghi e plotttiamo la distribuzione
full_saga_df['Dialogue Length'] = full_saga_df['Dialogue'].apply(lambda x: len(x))
plt.figure(figsize=(12, 6))
sns.histplot(full_saga_df['Dialogue Length'], bins=50)
plt.title('Distribuzione della lunghezza dei dialoghi')
plt.xlabel('Lunghezza del dialogo (numero di caratteri)')
plt.ylabel('Frequenza')
plt.show()

In [ ]:
full_saga_df

In [ ]:
full_saga_df['Dialogue Length'].describe()

Dato che la massima lunghezza è di 512 tokens, dobbiamo decidere come agire: scartare tutte le frase sopra una certa lunghezza? Aggiungiamo padding alle frasi corte e tronchiamo quelle lunghe. Data la distribuzione delle frasi possiamo scegliere come fixed lenght anche meno di 512. 

In [ ]:
model_name = 'bhadresh-savani/roberta-base-emotion'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(model_name, force_download=True,)
max_tokens_length = 256

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name)
print(model.config)
print(config.num_labels)  
print(config.id2label)    


Iniziamo il task di sentiment prediction. Con Roberta facciamo l'encoding dei dialoghi e chiediamo al modello di fare la previsione del sentimento. Ci sono 6 possibili emozioni prevedibili .

In [ ]:
# funzione per il calcolo delle emozioni per ogni dialogo. Al modello viene chiesto di tokenizzare ogni sequenza aggiungendo 
# il padding per le sequenze brevi e troncando le sequenze più lunghe di 256 caratteri. Questo numero è stato scelto perchè
# i dialoghi sopra questa lunghezza sono molto rari. 
# Infine applicando la funzione softmax ai logits calcolati e otteniamo la stima delle emozioni.

def calculate_emotion_score(dialogues, tokenizer, max_length=256):
    predictions = []

    for dialogue in tqdm(dialogues, desc="Processing dialogues", total=len(dialogues)):
        encoded = tokenizer.encode_plus(
            dialogue,
            add_special_tokens=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors='tf'
        )

        logits = model(**encoded).logits
        probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
        predictions.append(probs)
        
    return np.array(predictions)


La seguente cella chiama la funzione per il calcolo delle emozioni sull'intero set di copioni ed è discretamente oneroso. Per questo motivo il risultato verrà esportato in csv e all'occorrenza caricato.

In [ ]:
#emotions_scores = calculate_emotion_score(full_saga_df['Dialogue'], tokenizer, max_length=max_tokens_length)
labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

# Aggiungiamo le emozioni al DataFrame
#emotions_df = pd.DataFrame(emotions_scores, columns=labels)

# Aggiungiamo le emozioni al DataFrame principale
#full_saga_df = pd.concat([full_saga_df, emotions_df], axis=1)

# esportiamo in CSV
#full_saga_df = pd.read_csv("outputs/full_saga_with_emotions.csv", encoding='utf-8-sig')


In [ ]:
# importiamo il CSV
full_saga_df = pd.read_csv("outputs/full_saga_with_emotions.csv", encoding='utf-8-sig')
emotions_df = full_saga_df[labels]
emotions_df

#### Visualizzazione risultati

1. studiamo le statistiche del dataset.

In [ ]:
# studiamo emotions_df
emotions_df.describe()

2. sommiamo le colonne delle emozioni e visualizziamole

In [ ]:
# Somma delle emozioni
emotion_sums = emotions_df.sum().sort_values(ascending=False)

emotion_colors = {
    'joy': '#FFD700',        
    'sadness': '#1E90FF',    
    'anger': '#FF4500',      
    'fear': '#8B008B',       
    'surprise': '#00CED1',   
    'love': '#FF69B4',       
    'optimism': '#32CD32'    
}

# Ordinamento coerente
colors = [emotion_colors[emotion] for emotion in emotion_sums.index]

# Creazione grafico
plt.figure(figsize=(10, 6))
plt.bar(emotion_sums.index, emotion_sums.values, color=colors, edgecolor='black')

plt.title("Somma delle emozioni nei dialoghi")
plt.ylabel("Somma dei punteggi")
plt.xlabel("Emozioni")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


3. facciamo un distinguo basato sulle casate con diversi grafici.

In [ ]:
# adesso facciamo un altro grafico a barre con la somma delle emozioni, ma questa volta includiamo la suddivisione per case
gryffindor = full_saga_df[full_saga_df['House'] == 'Gryffindor']
hufflepuff = full_saga_df[full_saga_df['House'] == 'Hufflepuff']
ravenclaw = full_saga_df[full_saga_df['House'] == 'Ravenclaw']      
slytherin = full_saga_df[full_saga_df['House'] == 'Slytherin']
others = full_saga_df[~full_saga_df['House'].isin(['Gryffindor', 'Hufflepuff', 'Ravenclaw', 'Slytherin'])]

# Raggruppa e riordina le emozioni per colore (opzionale)
house_emotion_sums = full_saga_df.groupby('House').sum()  
house_emotion_sums = house_emotion_sums[labels]
house_emotion_sums.drop(['Beauxbatons Academy of Magic', 'Durmstrang Institute'], inplace=True)

# Colori per ogni casata
house_colors = {
    'Gryffindor': '#9C1F1F',
    'Hufflepuff': '#F0C330',
    'Ravenclaw': '#0E4C92',
    'Slytherin': '#1A472A',
    'Altri': '#999999'  # per personaggi senza casata
}

display(house_emotion_sums)


# Creazione grafico
house_emotion_sums.plot(kind='bar', stacked=True, figsize=(12, 6), color=colors)

plt.title("Somma delle emozioni (stacked) per casata")
plt.ylabel("Somma")
plt.xticks(rotation=45)
plt.legend(title="Emozione")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


Abbiamo il problema dello sbilanciamento. Passiamo comunque intanto da una visualizzazione assoluta ad una relativa

In [ ]:
relative_house_emotions = house_emotion_sums.apply(lambda row: row / row.sum(), axis=1)


relative_house_emotions

In [ ]:
# Creazione grafico
relative_house_emotions.plot(kind='bar', stacked=True, figsize=(12, 6), color=colors)

plt.title("Somma delle emozioni (stacked) per casata")
plt.ylabel("Somma")
plt.xticks(rotation=45)
plt.legend(title="Emozione")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

Tentiamo anche una visualizzazione su boxplot sui valori originali forniti dal transformer non normalizzati

In [ ]:

# Supponiamo che le tue colonne emozioni si chiamino così:
emotion_cols = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
df_filtered = full_saga_df[~full_saga_df['House'].isin(['Beauxbatons Academy of Magic', 'Durmstrang Institute'])]

# Passaggio 1: melt per fare il DataFrame "long"
df_long = pd.melt(df_filtered,
                  id_vars=['House'],    # mantieni questa colonna fissa
                  value_vars=emotion_cols,  # colonne da trasformare in riga
                  var_name='Emotion',
                  value_name='Score')

# Passaggio 2: plot boxplot
plt.figure(figsize=(14, 7))
sns.boxplot(data=df_long, x='Emotion', y='Score', hue='House')

plt.title("Distribuzione delle emozioni per casa (Boxplot)")
plt.ylabel("Percentuale emozione")
plt.xticks(rotation=45)
plt.legend(title='Casa', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 3.2 Approccio con i Lexicon
Proviamo ora invece un approccio più tradizionale. Usiamo il lexicon NRC Emotion

In [ ]:
lexicon_df = pd.read_excel("lexicon/NRCEmotionLexicon.xlsx", engine="openpyxl")

Riprendiamo il dataset ed eliminiamo le valutazioni precedenti

In [ ]:
full_saga_df = full_saga_df.drop(['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'], axis=1)
full_saga_df

In [ ]:
lexicon = {}

for word, anger, anticipation, disgust, fear, joy, sadness, surprise, trust in zip(lexicon_df["English Word"], lexicon_df["Anger"], lexicon_df["Anticipation"], lexicon_df["Disgust"], lexicon_df["Fear"], lexicon_df["Joy"], lexicon_df["Sadness"], lexicon_df["Surprise"], lexicon_df["Trust"]):
  lexicon[str(word).lower()] = {
    'anticipation' : anticipation,
    'disgust' : disgust,
    'fear' : fear,
    'joy' : joy,
    'sadness' : sadness,
    'surprise' : surprise,
    'trust' : trust,
  }

print(lexicon)

In [ ]:
def sentiment_score(sentence):
  anticipation = 0
  disgust = 0
  fear = 0
  joy = 0
  sadness = 0
  surprise = 0
  trust = 0
  for word in sentence.split(): 
    feelings = lexicon.get(word.lower())
    if (feelings is not None):
      anticipation += feelings.get("anticipation")
      disgust += feelings.get("disgust")
      fear += feelings.get("fear")
      joy += feelings.get("joy")
      sadness += feelings.get("sadness")
      surprise += feelings.get("surprise")
      trust += feelings.get("trust")
  return anticipation, disgust, fear, joy, sadness, surprise, trust

In [ ]:
full_saga_df['anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust'] = full_saga_df['Dialogue'].apply(sentiment_score)

In [ ]:
full_saga_df

In [ ]:
nrc_emotions = full_saga_df.drop(columns=['Chapter ID', 'Character ID', 'Dialogue', 'Character Name', 'Species', 'Gender', 'Dialogue Length'], axis=1)

nrc_emotions

In [ ]:
# manipoliamo il dataset per estrarre le emozioni

feelings_list = nrc_emotions.iloc[:, -1].tolist()
feelings_list

In [ ]:
anticipation_feeling = pd.Series(t[0] for t in feelings_list)
disgust_feeling = pd.Series(t[1] for t in feelings_list)
fear_feeling = pd.Series(t[2] for t in feelings_list)
joy_feeling = pd.Series(t[3] for t in feelings_list)
sadness_feeling = pd.Series(t[4] for t in feelings_list)
surprise_feeling = pd.Series(t[5] for t in feelings_list)
trust_feeling = pd.Series(t[6] for t in feelings_list)


nrc_emotions['anticipation'] = anticipation_feeling
nrc_emotions['disgust'] = disgust_feeling
nrc_emotions['fear'] = fear_feeling
nrc_emotions['joy'] = joy_feeling
nrc_emotions['sadness'] = sadness_feeling
nrc_emotions['surprise'] = surprise_feeling
nrc_emotions['trust'] = trust_feeling


nrc_emotions = nrc_emotions[['House', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']]

nrc_emotions


In [ ]:
nrc_emotions.describe()

In [ ]:
# Somma delle emozioni
sum_nrc_emotions = nrc_emotions[['anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']].sum().sort_values(ascending=False)

emotion_colors = {
    'anticipation': '#FFD700',  # oro – attesa, eccitazione
    'disgust': '#556B2F',       # verde oliva scuro – repulsione
    'fear': '#8B0000',          # rosso scuro – paura
    'joy': '#FFA500',           # arancione brillante – felicità
    'sadness': '#1E90FF',       # blu acceso – tristezza
    'surprise': '#BA55D3',      # viola medio – sorpresa
    'trust': '#228B22', 
}

# Ordinamento coerente
colors = [emotion_colors[emotion] for emotion in sum_nrc_emotions.index]

# Creazione grafico
plt.figure(figsize=(10, 6))
plt.bar(sum_nrc_emotions.index, sum_nrc_emotions.values, color=colors, edgecolor='black')

plt.title("Somma delle emozioni nei dialoghi")
plt.ylabel("Somma dei punteggi")
plt.xlabel("Emozioni")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# adesso facciamo un altro grafico a barre con la somma delle emozioni, ma questa volta includiamo la suddivisione per case
gryffindor = nrc_emotions[nrc_emotions['House'] == 'Gryffindor']
hufflepuff = nrc_emotions[nrc_emotions['House'] == 'Hufflepuff']
ravenclaw = nrc_emotions[nrc_emotions['House'] == 'Ravenclaw']      
slytherin = nrc_emotions[nrc_emotions['House'] == 'Slytherin']
labels = ['anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']


# Raggruppa e riordina le emozioni per colore (opzionale)
nrc_emotions_sum = nrc_emotions.groupby('House').sum()  # transponi: emozioni per riga
nrc_emotions_sum = nrc_emotions_sum[labels]
nrc_emotions_sum.drop(['Beauxbatons Academy of Magic', 'Durmstrang Institute'], inplace=True)

# Colori per ogni casata
house_colors = {
    'Gryffindor': '#9C1F1F',
    'Hufflepuff': '#F0C330',
    'Ravenclaw': '#0E4C92',
    'Slytherin': '#1A472A',
    'Altri': '#999999'  # per personaggi senza casata
}

display(nrc_emotions_sum)


# Creazione grafico
nrc_emotions_sum.plot(kind='bar', stacked=True, figsize=(12, 6), color=colors)

plt.title("Somma delle emozioni (stacked) per casata")
plt.ylabel("Somma")
plt.xticks(rotation=45)
plt.legend(title="Emozione")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
relative_nrc_emotions = nrc_emotions_sum.apply(lambda row: row / row.sum(), axis=1)


relative_nrc_emotions

In [ ]:
# Creazione grafico
relative_nrc_emotions.plot(kind='bar', stacked=True, figsize=(12, 6), color=colors)

plt.title("Somma delle emozioni (stacked) per casata")
plt.ylabel("Somma")
plt.xticks(rotation=45)
plt.legend(title="Emozione")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

Non pare emergere un particolare pattern che metta in evidenza il rapporto tra la casa e l'emozione dominante.
#### Conclusione
A causa del problema dello sbilanciamento e dell'assenza di pattern si può concludere che non è una buona idea fare una campagna di merketing di merchandising delle case basato sulla emozioni.

# Parte 4

In [184]:
from tqdm import tqdm
import pandas as pd
from transformers import pipeline

# --- 1. Estrazione emozioni + intensità (come già fai tu) ---
emotion_model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)
batch_size = 32

texts = dialogs['Dialogue'].tolist()
emotions = []
intensities = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    preds = emotion_model(batch)
    for pred in preds:
        pred_best = max(pred, key=lambda x: x['score'])
        label = pred_best['label'].lower()
        score = pred_best['score']
        emotions.append(label)
        intensities.append(score)

label_map = {
    'joy': 'joy',
    'anger': 'anger',
    'fear': 'fear',
    'sadness': 'sadness',
    'love': 'love',
    'surprise': 'surprise',
}

dialogs['Emotion'] = pd.Series(emotions).map(label_map)
dialogs['Intensity'] = intensities
dialogs = dialogs[dialogs['Emotion'].isin(label_map.values())]
dialogs['Emotion'] = pd.Categorical(dialogs['Emotion'], categories=list(label_map.values()))
dialogs = dialogs.rename(columns={'speaker': 'Character'}) # opzionale

# --- 2. Calcola top 5 capitoli per ogni emozione (usando anche Movie ID) ---
grouped = dialogs.groupby(['Chapter ID', 'Movie ID', 'Emotion'])['Intensity'].mean().reset_index()

top_chapters_list = []
for emo in grouped['Emotion'].unique():
    top = grouped[grouped['Emotion'] == emo].sort_values(by='Intensity', ascending=False).head(5)
    top_chapters_list.extend([
        (row['Chapter ID'], row['Movie ID'], emo, row['Intensity'])
        for _, row in top.iterrows()
    ])

top_chapters_df = pd.DataFrame(top_chapters_list, columns=['Chapter ID', 'Movie ID', 'Emotion', 'Intensity'])

# --- OUTPUT INTERMEDIO: stampa lista scene top per emozione ---
print("\n=== Top 5 scene per ogni emozione ===\n")
for emo in top_chapters_df['Emotion'].unique():
    df_emo = top_chapters_df[top_chapters_df['Emotion'] == emo]
    print(f"Emozione: {emo}")
    for _, row in df_emo.iterrows():
        print(f"  Film: {row['Movie ID']} | Capitolo: {row['Chapter ID']} | Intensità media: {row['Intensity']:.2f}")
    print()


Device set to use cpu
100%|██████████| 233/233 [06:26<00:00,  1.66s/it]


=== Top 5 scene per ogni emozione ===

Emozione: joy
  Film: 1 | Capitolo: 24 | Intensità media: 0.99
  Film: 4 | Capitolo: 106 | Intensità media: 0.99
  Film: 4 | Capitolo: 129 | Intensità media: 0.98
  Film: 3 | Capitolo: 77 | Intensità media: 0.97
  Film: 6 | Capitolo: 176 | Intensità media: 0.97

Emozione: anger
  Film: 3 | Capitolo: 83 | Intensità media: 0.96
  Film: 1 | Capitolo: 4 | Intensità media: 0.89
  Film: 7 | Capitolo: 207 | Intensità media: 0.88
  Film: 3 | Capitolo: 86 | Intensità media: 0.86
  Film: 7 | Capitolo: 205 | Intensità media: 0.86

Emozione: fear
  Film: 1 | Capitolo: 5 | Intensità media: 0.98
  Film: 2 | Capitolo: 61 | Intensità media: 0.98
  Film: 5 | Capitolo: 134 | Intensità media: 0.98
  Film: 2 | Capitolo: 40 | Intensità media: 0.98
  Film: 8 | Capitolo: 210 | Intensità media: 0.98

Emozione: sadness
  Film: 4 | Capitolo: 121 | Intensità media: 0.97
  Film: 3 | Capitolo: 102 | Intensità media: 0.95
  Film: 3 | Capitolo: 97 | Intensità media: 0.95
  Fil


C:\Users\Utente\AppData\Local\Temp\ipykernel_36808\4210539772.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dialogs['Emotion'] = pd.Categorical(dialogs['Emotion'], categories=list(label_map.values()))
C:\Users\Utente\AppData\Local\Temp\ipykernel_36808\4210539772.py:39: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = dialogs.groupby(['Chapter ID', 'Movie ID', 'Emotion'])['Intensity'].mean().reset_index()


In [201]:
!pip install transformers torch
!pip install accelerate

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [207]:
import pandas as pd
from tqdm import tqdm
from transformers import pipeline

# --- 1. Prepara i testi concatenati per ogni capitolo ---
chapter_texts = dialogs.groupby(['Chapter ID', 'Movie ID'])['Dialogue'].apply(lambda x: " ".join(x)).reset_index()

# --- 2. Merge pulito per avere il testo unico da riassumere ---
for col in ['dialogue', 'dialogue_x', 'dialogue_y', 'Dialogue_x', 'Dialogue_y']:
    if col in top_chapters_df.columns:
        top_chapters_df = top_chapters_df.drop(columns=col)
top_chapters_df = top_chapters_df.merge(chapter_texts, on=['Chapter ID', 'Movie ID'], how='left')

# --- 3. Output intermedio: mostra le scene che saranno riassunte ---
print("\n=== SCENE SCELTE PER EMOZIONE ===\n")
for emo in top_chapters_df['Emotion'].unique():
    df_emo = top_chapters_df[top_chapters_df['Emotion'] == emo]
    print(f"Emozione: {emo}")
    for _, row in df_emo.iterrows():
        print(f"  Film: {row['Movie ID']} | Capitolo: {row['Chapter ID']} | Intensità media: {row['Intensity']:.2f}")
    print()

# --- 4. Setup HuggingFace summarizer ---
# Puoi usare anche modelli diversi, per esempio: "facebook/bart-large-cnn", "philschmid/bart-large-cnn-samsum"
summarizer = pipeline("summarization", model="philschmid/bart-large-cnn-samsum")

def hf_summarize_dialogues(dialogue_text, max_tokens=110):
    # prompt = f"{dialogue_text}\n\nSummary of what happens:"
    prompt = dialogue_text[:2000]  # Solo dialoghi
    try:
        result = summarizer(prompt, max_length=max_tokens, min_length=30, do_sample=False)[0]['summary_text']
        # Pulizia post-processing
        for pattern in [
            "Summarize a scene from Harry Potter based on the following dialogues. Do not repeat the dialogues, but reconstruct what happens in the scene.",
            "Summarize this scene from Harry Potter, based only on the following dialogues. Write a short narrative summary (3-4 lines), mentioning the characters and actions. Do not repeat the dialogues, but reconstruct what happens in the scene."
        ]:
            result = result.replace(pattern, "").strip()
        return result
    except Exception as e:
        print("Errore HuggingFace:", e)
        return "N/A"

# --- 5. Genera i riassunti SOLO per i capitoli che hanno testo ---
summaries = []
for i, row in tqdm(top_chapters_df.iterrows(), total=len(top_chapters_df)):
    chapter_id = row['Chapter ID']
    movie = row['Movie ID']
    emo = row['Emotion']
    intensity = row['Intensity']
    dialogues_text = row.get('Dialogue', row.get('dialogue', ""))
    if not isinstance(dialogues_text, str) or dialogues_text.strip() == "":
        summary = "N/A (no dialogues available for this chapter)"
    else:
        summary = hf_summarize_dialogues(dialogues_text)
    summaries.append({
        'Film': movie,
        'Capitolo': chapter_id,
        'Emozione': emo,
        'Intensità_media': round(float(intensity), 2),
        'Riassunto_narrativo': summary
    })

# --- 6. Risultato finale in DataFrame
final_df = pd.DataFrame(summaries)
print("\n=== Tabella finale riassunti top scene per emozione ===\n")
print(final_df[['Film', 'Capitolo', 'Emozione', 'Intensità_media', 'Riassunto_narrativo']])

# (Opzionale: salva su CSV)
# final_df.to_csv("output_top_scene_riassunti.csv", index=False)



=== SCENE SCELTE PER EMOZIONE ===

Emozione: joy
  Film: 1 | Capitolo: 24 | Intensità media: 0.99
  Film: 4 | Capitolo: 106 | Intensità media: 0.99
  Film: 4 | Capitolo: 129 | Intensità media: 0.98
  Film: 3 | Capitolo: 77 | Intensità media: 0.97
  Film: 6 | Capitolo: 176 | Intensità media: 0.97

Emozione: anger
  Film: 3 | Capitolo: 83 | Intensità media: 0.96
  Film: 1 | Capitolo: 4 | Intensità media: 0.89
  Film: 7 | Capitolo: 207 | Intensità media: 0.88
  Film: 3 | Capitolo: 86 | Intensità media: 0.86
  Film: 7 | Capitolo: 205 | Intensità media: 0.86

Emozione: fear
  Film: 1 | Capitolo: 5 | Intensità media: 0.98
  Film: 2 | Capitolo: 61 | Intensità media: 0.98
  Film: 5 | Capitolo: 134 | Intensità media: 0.98
  Film: 2 | Capitolo: 40 | Intensità media: 0.98
  Film: 8 | Capitolo: 210 | Intensità media: 0.98

Emozione: sadness
  Film: 4 | Capitolo: 121 | Intensità media: 0.97
  Film: 3 | Capitolo: 102 | Intensità media: 0.95
  Film: 3 | Capitolo: 97 | Intensità media: 0.95
  Film: 5

Device set to use cpu
100%|██████████| 30/30 [08:27<00:00, 16.93s/it]


=== Tabella finale riassunti top scene per emozione ===

    Film  Capitolo  Emozione  Intensità_media  \
0      1        24       joy             0.99   
1      4       106       joy             0.99   
2      4       129       joy             0.98   
3      3        77       joy             0.97   
4      6       176       joy             0.97   
5      3        83     anger             0.96   
6      1         4     anger             0.89   
7      7       207     anger             0.88   
8      3        86     anger             0.86   
9      7       205     anger             0.86   
10     1         5      fear             0.98   
11     2        61      fear             0.98   
12     5       134      fear             0.98   
13     2        40      fear             0.98   
14     8       210      fear             0.98   
15     4       121   sadness             0.97   
16     3       102   sadness             0.95   
17     3        97   sadness             0.95   
18     5   

In [208]:
from IPython.display import display, HTML

for emo in final_df['Emozione'].unique():
    df_emo = final_df[final_df['Emozione'] == emo][['Film', 'Capitolo', 'Intensità_media', 'Riassunto_narrativo']]
    display(HTML(f"<h2 style='color: #3874a7'>{emo.capitalize()}</h2>"))
    display(df_emo.style.set_properties(**{
        'border': '1px solid #999',
        'padding': '8px',
        'text-align': 'left'
    }))


,Film,Capitolo,Intensità_media,Riassunto_narrativo
0,1,24,0.990000,The Philosopher's Stone is guarded by Fluffy on the 3rd floor under the trapdoor. Nicholas Flamel is the only known maker of the Stone. Hagrid won it off a stranger at a pub.
1,4,106,0.990000,The final of the Quidditch World Cup is taking place. Ireland are playing against the UK. Dad is looking forward to the match.
2,4,129,0.980000,Alastor Moody stole gilliweed from Albus' store. He will be welcomed back to Hogwarts like a hero.
3,3,77,0.970000,Professor Lupin will be the new Defense Against the Dark Arts teacher at Hogwarts. Rubeus Hagrid will replace Professor Lupin as the Care of Magical Creatures teacher. Hogwarts will be host to the dementors of Azkaban until such a time as Sirius Black is captured.
4,6,176,0.970000,Ron broke up with Lavender. Lavender visited Ron in the hospital and they had a long conversation. Ron doesn't remember anything from that night.


,Film,Capitolo,Intensità_media,Riassunto_narrativo
5,3,83,0.960000,Professor Potter's professor is incapable of teaching at the present time. Mr. Malfoy wants to know when she came in and who can tell him the difference between Animagus and a werewolf.
6,1,4,0.890000,Harry Potter has been accepted at Hogwarts' School of Witchcraft and Wizardry. Harry's sister was a witch and his parents died in a car crash. Dursley is angry and wants Harry to leave.
7,7,207,0.880000,Dobby is at Shell Cottage on the outskirts of Tinworth. He will meet Draco at the top of the stairs in 10 seconds.
8,3,86,0.860000,Malfoy and Weasle-Bee are shopping for a new house in the Shrieking Shack. It's meant to be the most haunted building in Britain.
9,7,205,0.860000,They call her a Stinging jinx. She's not related to Arthur Weasley. He's ten times more powerful than her.


,Film,Capitolo,Intensità_media,Riassunto_narrativo
10,1,5,0.980000,Professor Quirrell will be Harry Potter's Defense Against the Dark Arts teacher at Hogwarts. Doris Crockford is happy to see Harry. Hagrid wonders why Harry is famous.
11,2,61,0.980000,"The flying gear jammed, so Arania and the others had to follow the spiders to get out of Azkaban. They are glad to be out of there."
12,5,134,0.980000,"Hermione and Harry haven't written to each other for a long time. The Ministry is furious with Harry, because he's telling people about Voldemort and the Order everyone's talking about."
13,2,40,0.980000,Harry Potter's autobiography is celebrating its 27th week atop the Daily Prophet bestseller list. He bought it in Flourish and Blotts this morning.
14,8,210,0.980000,Harry left Gringotts because the Death Eaters will control it soon. Bellatrix was terrified when she thought they'd been in the vault and she kept asking Harry what else they'd taken. Harry suspects there's a Horcrux there.


,Film,Capitolo,Intensità_media,Riassunto_narrativo
15,4,121,0.970000,Mr Diggory has won the task. Miss Delacour has been forced to retire. Harry saved her even though she wasn't his sister.
16,3,102,0.950000,"Harry has resigned from his job, but Ron thinks he's been sacked. Ron is confused as to why Harry looks so miserable."
17,3,97,0.950000,"Sirius Black is in the topmost cell of the Dark Tower. If Hermione doesn't return before the last chime, she will be killed."
18,5,138,0.940000,"Harry, Ron and Hermione have been named House Prefects. Harry didn't get a letter. Harry and Ron will go back downstairs to enjoy themselves."
19,2,51,0.940000,Lucius Malfoy taught Draco how to open the Chamber of Secrets. Professor Dumbledore has granted him permission to start a Dueling Club. Professor Snape will help him with a short demonstration.


,Film,Capitolo,Intensità_media,Riassunto_narrativo
20,1,1,nan,Professor McGonagall would trust Hagrid with her life. Hagrid is the only family he has. It's not really goodbye after all.
21,2,1,nan,N/A (no dialogues available for this chapter)
22,3,1,nan,N/A (no dialogues available for this chapter)
23,4,1,nan,N/A (no dialogues available for this chapter)
24,5,1,nan,N/A (no dialogues available for this chapter)


,Film,Capitolo,Intensità_media,Riassunto_narrativo
25,1,11,0.920000,"Trevor will follow First years to the boats, because he needs to hurry up. Trevor will be late, because First Years are running late."
26,7,196,0.920000,"It's good to know your enchantments work. It's a good thing that the smell doesn't bother you, because it's bad."
27,2,67,0.890000,Ginny hurt Harry. Riddle made her do it. Ron helped her escape. Harry and Ron will find her soon.
28,6,174,0.850000,Professor Merrythought is retiring. Harry was in the library the other night and read something odd in the Restricted section. Professor Merrythought tampered with his own memory.
29,3,85,0.830000,He's trying to get to Hogsmeade. He wants to get away from everyone. He's not sure where he got it.
